# Week 18: Autonomous AI Agents & Multi-Agent Systems
## Notebook 01: Tree of Thoughts (ToT) & Deliberate Reasoning Search from Scratch

### 1. Conceptual Intuition & Real-World Use Case
Standard LLM prompting generates responses in a single autoregressive linear pass:
- **Input-Output (IO):** Single prompt $\to$ single output.
- **Chain-of-Thought (CoT):** Linear step-by-step reasoning ($s_1 \to s_2 \to s_3$). If step 2 contains an error, the model cannot backtrack and cascades into failure.

In 2023, Yao et al. (Princeton/Google DeepMind) introduced **Tree of Thoughts (ToT)**:
- Enables deliberate decision-making by considering multiple reasoning paths.
- Evaluates intermediate states heuristically ($V(s)$).
- Actively **backtracks** and prunes dead-end paths using classical search algorithms (BFS / DFS / A*), directly laying the groundwork for modern reasoning models (e.g. OpenAI o1).

---

### 2. Mathematical Foundations
A problem is formulated as a search tree over states $s = [x, z_{1 \dots i}]$:
1. **Thought Generation:** $G(p_\theta, s, k)$ generates $k$ candidate next steps.
2. **State Evaluation:** A heuristic scoring function $V(p_\theta, S)$ scores state viability:
   $$V(s) \in [0, 1] \quad \text{or} \quad \{\text{sure}, \text{maybe}, \text{impossible}\}$$
3. **Search Algorithm (BFS with Beam Pruning):**
   At step $t$, select the top $b$ highest-scoring candidate states $S_t \subset C_t$ and discard the rest.

In [ ]:
import itertools

# Problem: Game of 24 (Reach target 24 using numbers [4, 4, 6, 8] with +, -, *, /)
target = 24
initial_numbers = [4, 4, 6, 8]

class ToTSolver:
    def __init__(self, target=24, beam_width=3):
        self.target = target
        self.beam_width = beam_width

    def generate_next_states(self, state):
        # A state is a tuple of available numbers and the history of operations
        numbers, history = state
        next_states = []
        
        # Pick any pair of numbers to combine
        for i, j in itertools.combinations(range(len(numbers)), 2):
            a, b = numbers[i], numbers[j]
            remaining = [numbers[k] for k in range(len(numbers)) if k != i and k != j]
            
            # Possible operations
            ops = [
                (a + b, f"({a} + {b})"),
                (a * b, f"({a} * {b})"),
                (a - b, f"({a} - {b})"),
                (b - a, f"({b} - {a})")
            ]
            if b != 0 and a % b == 0:
                ops.append((a // b, f"({a} / {b})"))
            if a != 0 and b % a == 0:
                ops.append((b // a, f"({b} / {a})"))
                
            for result, expr in ops:
                next_states.append((remaining + [result], history + [expr]))
        return next_states

    def evaluate_state(self, state):
        numbers, _ = state
        if len(numbers) == 1:
            return 1.0 if numbers[0] == self.target else 0.0
        # Heuristic: Can 24 be reached roughly from remaining numbers?
        # Penalize states where numbers grow excessively large (> 100)
        if any(abs(n) > 100 for n in numbers):
            return 0.1
        return 0.8  # viable candidate

    def solve(self, start_numbers):
        # Queue of active states: (numbers, history)
        current_beam = [(start_numbers, [])]
        
        depth = 0
        while current_beam:
            depth += 1
            candidates = []
            for state in current_beam:
                # Check goal condition
                if len(state[0]) == 1 and state[0][0] == self.target:
                    return state[1]  # Solution found!
                
                # Expand
                for nxt in self.generate_next_states(state):
                    score = self.evaluate_state(nxt)
                    if score > 0.0:
                        candidates.append((score, nxt))
            
            if not candidates:
                break
                
            # Beam search: keep top-b candidates
            candidates.sort(key=lambda x: x[0], reverse=True)
            current_beam = [item[1] for item in candidates[:self.beam_width]]
            
        return None

solver = ToTSolver(target=24, beam_width=5)
solution_steps = solver.solve(initial_numbers)

print(f"Goal: Make 24 using {initial_numbers}")
if solution_steps:
    print("Found Valid Solution Steps via Tree Search:")
    for step_num, op in enumerate(solution_steps, 1):
        print(f"  Step {step_num}: {op}")
else:
    print("No solution found within search beam.")

### 3. Step-by-Step Code Walkthrough
1. **State Expansion:** At each reasoning node, pairs of numbers are combined using algebraic operations.
2. **Heuristic Evaluation:** States producing invalid outputs or exceeding limits are pruned.
3. **Backtracking via Beam Search:** The solver maintains the top-$k$ most promising partial reasoning branches, abandoning dead ends.

---

### 4. Industry Applications & Trade-offs
- **Complex Software Architecture Planning:** Writing multi-file software projects where an architectural decision in File 1 affects imports in File 10.
- **Mathematical & Symbolic Theorem Proving:** Searching through formal proof graphs (Lean, Isabelle).

---

### 5. Top Interview Questions & Answers
**Q1: How does Tree of Thoughts fundamentally differ from Chain-of-Thought (CoT)?**
> **Answer:** CoT generates a single irreversible sequential trajectory of reasoning tokens. ToT models problem solving as exploration over a graph/tree of thoughts, allowing branching, self-evaluation of partial thoughts, and backtracking when a path fails.

**Q2: What is the relationship between Tree of Thoughts and Monte Carlo Tree Search (MCTS)?**
> **Answer:** ToT uses classical search (BFS/DFS) with heuristic or LLM-evaluated state scores. MCTS extends this by balancing exploration vs exploitation (via UCT bounds) and running rollouts or value-network evaluations, which is the architectural foundation of reasoning engines like AlphaGo and OpenAI o1.